importation des libs et chargement des datas

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px  # Interface haut niveau pour graphiques simples
import plotly.graph_objects as go  # Interface bas niveau pour contrôle précis
import seaborn as sns
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots  # Création de grilles de graphiques

#Affichage complet des colonnes et des lignes (pas de retour a la ligne)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.expand_frame_repr", False)

#Import du fichier csv customer
url = "./../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url,sep=",",skipinitialspace=True)

# Affichage des dimension du jeu de données
print(f"Le jeu de données a {df.shape[0]} lignes et {df.shape[1]} colonnes")

# Affichez les 5 premières lignes
print(df.head())




Le jeu de données a 7043 lignes et 21 colonnes
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges  TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85         29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.50    No
2  3668-QPYBK    Male              0      No         No       2          Yes 

Affichez les types de données de chaque colonne

Nombre pour chaque variable

In [3]:
# Affichez les types de données de chaque colonne
display(df.info())
print(" " * 50)
print("-" * 50)
print(" " * 50)
#Nombre de manquants pour chaque colonnes

if (df.isna().sum() == 0).all():
    print("Il ne manque aucune valeur.")
else:
    print("Il y a des valeurs manquantes.")
    manquant=df.isna().sum()
    print(f"{manquant[manquant != 0]}")  

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

None

                                                  
--------------------------------------------------
                                                  
Il y a des valeurs manquantes.
TotalCharges    11
dtype: int64


Recherche de doublon

In [4]:
#Recherhce de doublon
print(f"Il y a {df.duplicated().sum()} doublons dans le jeu de données.")
if df.duplicated().sum() > 0:
    print("Voici les lignes en double :")
    print(df[df.duplicated()])

Il y a 0 doublons dans le jeu de données.


Recherche des outlier

In [5]:
# Création d'une figure avec 2 sous-graphiques côte à côte
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=('SeniorCitizen',
                    'Tenure',
                    'MonthlyCharges',
                    'TotalCharges')
)

# Ajout du premier boxplot (SeniorCitizen)
fig.add_trace(
    go.Box(y=df['SeniorCitizen'], name='SeniorCitizen', marker_color='#FF6B6B'),
    row=1, col=1
)

# Ajout du second boxplot (Tenure))
fig.add_trace(
    go.Box(y=df['tenure'], name='Tenure', marker_color='#4ECDC4'),
    row=1, col=2
)
# Ajout du troisieme boxplot (MonthlyCharges))
fig.add_trace(
    go.Box(y=df['MonthlyCharges'], name='MonthlyCharges', marker_color="#048620"),
    row=1, col=3
)
# Ajout du quatrieme boxplot (TotalCharges))
fig.add_trace(
    go.Box(y=df['TotalCharges'], name='TotalCharges', marker_color="#280462"),
    row=1, col=4
)

# Mise à jour de la mise en page
fig.update_layout(
    height=500,
    showlegend=False,
    title_text="Comparaison des distributions avec et sans outliers",
    title_x=0.5,
    title_font_size=16
)

# Ajout des labels pour les axes y
fig.update_yaxes(title_text="Nombre de vidéos", row=1, col=1)
fig.update_yaxes(title_text="Nombre de vidéos", row=1, col=2)
fig.update_yaxes(title_text="Nombre de vidéos", row=1, col=3)
fig.update_yaxes(title_text="Nombre de vidéos", row=1, col=4)

fig.show()

In [6]:
def detecter_outliers_iqr(df: pd.DataFrame, colonne: str, coefficient: float = 1.5) -> tuple:
    """
    Détecte les outliers dans une colonne numérique selon la méthode IQR.

    Cette fonction calcule les quartiles Q1 et Q3, puis l'écart interquartile (IQR).
    Elle identifie ensuite les valeurs qui dépassent les limites définies par
    Q1 - coefficient * IQR et Q3 + coefficient * IQR.

    Parameters
    ----------
    df : pd.DataFrame
        Le DataFrame contenant les données à analyser.
    colonne : str
        Le nom de la colonne numérique à analyser.
    coefficient : float, default=1.5
        Le coefficient multiplicateur de l'IQR pour définir les limites.
        La valeur standard est 1.5. Une valeur plus élevée (ex: 3) détectera
        uniquement les outliers extrêmes.

    Returns
    -------
    tuple
        Un tuple contenant :
        - list : Liste triée des index des outliers détectés
        - float : Limite inférieure (valeurs en dessous sont des outliers)
        - float : Limite supérieure (valeurs au-dessus sont des outliers)

    Examples
    --------
    >>> outliers_idx, limite_inf, limite_sup = detecter_outliers_iqr(df, 'Vidéos')
    >>> print(f"Outliers détectés aux index : {outliers_idx}")

    Notes
    -----
    Cette fonction utilise la méthode quantile() de Pandas pour calculer
    les quartiles (gère automatiquement les valeurs NaN).
    """

    # Calcul du 1er quartile (Q1) - 25% des données sont inférieures
    q1 = df[colonne].quantile(0.25)

    # Calcul du 3ème quartile (Q3) - 75% des données sont inférieures
    q3 = df[colonne].quantile(0.75)

    # Calcul de l'écart interquartile (IQR)
    iqr = q3 - q1

    # Calcul des limites
    limite_inferieure = q1 - coefficient * iqr
    limite_superieure = q3 + coefficient * iqr

    # Création d'un masque booléen pour identifier les outliers
    masque_outliers = (df[colonne] < limite_inferieure) | (df[colonne] > limite_superieure)

    # Extraction des index des outliers
    index_outliers = df[masque_outliers].index.tolist()

    return sorted(index_outliers), limite_inferieure, limite_superieure

# boucle sur les colonnes numleriques
for col in df.select_dtypes(include="number").columns:
    index_outliers, limite_inf, limite_sup = detecter_outliers_iqr(df, col)

    # Affichage des résultats
    print(f"colonne {col}")
    print(f"Les potentiels outliers (valeurs < {limite_inf:.2f} ou > {limite_sup:.2f}) sont aux index : ")
    print(index_outliers)

colonne SeniorCitizen
Les potentiels outliers (valeurs < 0.00 ou > 0.00) sont aux index : 
[20, 30, 31, 34, 50, 52, 53, 54, 55, 57, 72, 75, 78, 91, 99, 103, 113, 126, 129, 139, 140, 144, 168, 176, 177, 214, 238, 243, 244, 245, 247, 252, 259, 260, 261, 262, 267, 270, 273, 277, 288, 290, 293, 297, 301, 306, 308, 313, 318, 320, 326, 327, 328, 329, 340, 345, 349, 352, 354, 356, 358, 382, 385, 386, 391, 392, 394, 396, 398, 407, 410, 414, 419, 425, 426, 441, 451, 464, 466, 470, 476, 482, 493, 498, 501, 505, 510, 512, 517, 518, 521, 526, 529, 533, 548, 571, 574, 578, 582, 585, 609, 613, 617, 620, 627, 628, 629, 630, 638, 643, 645, 648, 649, 654, 674, 675, 683, 687, 694, 698, 707, 719, 724, 725, 728, 738, 739, 745, 746, 747, 755, 785, 800, 814, 826, 834, 835, 840, 862, 881, 886, 887, 892, 905, 910, 911, 915, 916, 924, 926, 933, 935, 947, 948, 950, 960, 962, 963, 965, 973, 978, 985, 997, 1001, 1005, 1009, 1022, 1023, 1025, 1026, 1029, 1032, 1040, 1050, 1061, 1069, 1078, 1081, 1085, 1086, 1088, 

Recherche de incoherances

In [7]:
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())
if inco_tenure != 0:
    print(f"il y a  {inco_tenure} valeur negatives dans tenure")
if inco_monthly != 0:
    print(f"il y a  {inco_monthly} valeur negatives dans MonthlyCharges")

In [8]:
#Recherche valeur autre que Yes No pour "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"

cols_yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn","MultipleLines","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","Contract",]
valeurs_MultipleLines = ["Yes", "No","No phone service"]
valeurs_Contract = ["Month-to-month", "One year","Two year"]
valeurs_Other = ["No internet service","Yes", "No"]

for col in cols_yes_no:
    s = df[col]
    #selection des valeurs non NaN qui ne sont pas dans les valeurs autorisées
    if col == "MultipleLines":
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_MultipleLines)]
    else:
        if col=="Contract":
            invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_Contract)]
        else:    
            #pour les colonnes autre que MultipleLines, les valeurs autorisées sont uniquement "Yes" et "No"
            invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_Other)]
        

    
    if invalides.empty:
        print(f"{col}: OK (NaN={s.isna().sum()})")
    else:
        print(f"{col}: INVALIDES -> {invalides.unique()} (NaN={s.isna().sum()})")



Partner: OK (NaN=0)
Dependents: OK (NaN=0)
PhoneService: OK (NaN=0)
PaperlessBilling: OK (NaN=0)
Churn: OK (NaN=0)
MultipleLines: OK (NaN=0)
OnlineSecurity: OK (NaN=0)
OnlineBackup: OK (NaN=0)
DeviceProtection: OK (NaN=0)
TechSupport: OK (NaN=0)
StreamingTV: OK (NaN=0)
StreamingMovies: OK (NaN=0)
Contract: OK (NaN=0)


Recap du controle qualité

In [9]:
# 1) Valeurs manquantes
missing_counts = df.isna().sum()
missing_total = int(missing_counts.sum())

# 2) Doublons (lignes complètes)
duplicates_count = int(df.duplicated().sum())

# 3) outliers



for col in df.select_dtypes(include="number").columns:
    index_outliers, limite_inf, limite_sup = detecter_outliers_iqr(df, col)
    if len(index_outliers) > 0:
        if len(index_outliers) <= 10:
            print(f"colonne {col}")
            print(f"Les potentiels outliers (valeurs < {limite_inf:.2f} ou > {limite_sup:.2f}) sont aux index : ")
            print(index_outliers)
        else:
            print(f"il y a {len(index_outliers)} outliers sur la colonne {col}")
            valeurs_outlier = np.unique(df.loc[index_outliers, col].values)
            print("les valeurs sont:")
            for val in valeurs_outlier:
                print([val])


# 4) Cohérences métier
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())


# 5) Résumé
print("=== Résumé qualité des données ===")
print(f"Valeurs manquantes totales : {missing_total}")
print(f"Doublons (lignes complètes) : {duplicates_count}")
print(f"Incohérences tenure < 0 : {inco_tenure}")
print(f"Incohérences MonthlyCharges < 0 : {inco_monthly}")


print("\nDétail valeurs manquantes (colonnes avec au moins 1 NaN) :")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

if (
    missing_total == 0
    and duplicates_count == 0
    and inco_tenure == 0
    and inco_monthly == 0
       
    ):
    print("\nConclusion : controles qualite OK.")
else:
    print("\nConclusion : il reste des points a traiter.")

il y a 1142 outliers sur la colonne SeniorCitizen
les valeurs sont:
[np.int64(1)]
=== Résumé qualité des données ===
Valeurs manquantes totales : 11
Doublons (lignes complètes) : 0
Incohérences tenure < 0 : 0
Incohérences MonthlyCharges < 0 : 0

Détail valeurs manquantes (colonnes avec au moins 1 NaN) :
TotalCharges    11
dtype: int64

Conclusion : il reste des points a traiter.


Exploration de la cible:
    taux Global de churn
    puis par segments 
        Contrat
        InternetService
        tenure
        Payment methode
        gender
        Partner
        Dependents
        PhoneService
        MultipleLines
        InternetService
        OnlineSecurity
        OnlineBackup
        DeviceProtection
        TechSupport
        StreamingTV
        StreamingMovies
        PaperlessBilling
        PaymentMethod
        MonthlyCharges

In [10]:
#Taux global de churn
taux_global_churn = (df["Churn"] == "Yes").mean()
print(f"Taux global de churn (Pourcentage de personne qui ont résilié) : {taux_global_churn:.2%}")

Taux global de churn (Pourcentage de personne qui ont résilié) : 26.54%


Definition du protocole d'evalutaion

Split : train / validation / test (stratifié  Churn)
Métriques : ROC-AUC, Precision, Recall, F1-score
Déséquilibre : prise en compte via l’analyse de ces métriques (notamment Recall/F1 sur churn)

In [16]:
def plot_churn_rate_bar(data, segment_col, title=None):
    """
    Calcule et affiche le taux de churn par segment sous forme de bar chart Plotly.

    La fonction s'appuie sur la colonne `Churn_num` (0 = non churn, 1 = churn),
    agrège par `segment_col`, convertit la moyenne en pourcentage, imprime le
    tableau résultant, puis affiche un graphique en barres trié du taux le plus
    élevé au plus faible.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame contenant au minimum les colonnes `Churn_num` et `segment_col`.
    segment_col : str
        Colonne de segmentation utilisée pour le groupement
        (ex. `Contract`, `PaymentMethod`, `tenure_segment`).
    title : str, optional
        Titre personnalisé du graphique. Si `None`, un titre par défaut est utilisé.

    Returns
    -------
    None
        La fonction affiche le tableau agrégé dans la console et le graphique Plotly.
    """
    
    
    
    
    tmp = (
        data.groupby(segment_col, dropna=False)["Churn_num"]
            .mean()
            .mul(100)
            .reset_index(name="churn_rate")
            .sort_values("churn_rate", ascending=False)
    )

    print(f"\n--- {segment_col} ---")
    print(tmp)

    fig = px.bar(
        tmp,
        x=segment_col,
        y="churn_rate",
        color=segment_col,  # une couleur différente par barre
        text=tmp["churn_rate"].round(1).astype(str) + "%",
        labels={segment_col: segment_col, "churn_rate": "Taux de churn (%)"},
        title=title or f"Taux de churn par {segment_col}"
    )

    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        xaxis_tickangle=-25,
        yaxis_range=[0, max(5, tmp["churn_rate"].max() * 1.15)]
    )
    fig.show()


# Churn yes/no -> 0/1
df["Churn_num"] = (df["Churn"]
                    .astype(str).str.strip().str.lower()
                    .map({"yes": 1, "no": 0,"1": 1, "0": 0,"true": 1, "false": 0
                    })
                    )



# Segment tenure
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"]
df["tenure_segment"] = pd.cut(df["tenure"], bins=bins, labels=labels, include_lowest=True)

# Génération de tous les graphes
segments = [
    "tenure",
    "tenure_segment",
    "Contract",
    "PaymentMethod",
    "PhoneService",
    "OnlineSecurity",
    "MultipleLines",
    "MonthlyCharges",
    "TotalCharges"
]

for col in segments:
    if col in df.columns:
        plot_churn_rate_bar(df, col, f"Taux de churn par {col}")
    else:
        print(f"Colonne absente: {col}")



--- tenure ---
    tenure  churn_rate
1        1   61.990212
2        2   51.680672
5        5   48.120301
4        4   47.159091
3        3   47.000000
..     ...         ...
63      63    5.555556
64      64    5.000000
71      71    3.529412
72      72    1.657459
0        0    0.000000

[73 rows x 2 columns]



--- tenure_segment ---
  tenure_segment  churn_rate
0           0-12   47.438243
1          13-24   28.710938
2          25-36   21.634615
3          37-48   19.028871
4          49-60   14.423077
5          61-72    6.609808



--- Contract ---
         Contract  churn_rate
0  Month-to-month   42.709677
1        One year   11.269518
2        Two year    2.831858



--- PaymentMethod ---
               PaymentMethod  churn_rate
2           Electronic check   45.285412
3               Mailed check   19.106700
0  Bank transfer (automatic)   16.709845
1    Credit card (automatic)   15.243101



--- PhoneService ---
  PhoneService  churn_rate
1          Yes   26.709637
0           No   24.926686



--- OnlineSecurity ---
        OnlineSecurity  churn_rate
0                   No   41.766724
2                  Yes   14.611194
1  No internet service    7.404980



--- MultipleLines ---
      MultipleLines  churn_rate
2               Yes   28.609896
0                No   25.044248
1  No phone service   24.926686



--- MonthlyCharges ---
      MonthlyCharges  churn_rate
1532          114.20       100.0
1576          117.45       100.0
1561          116.20       100.0
673            67.75       100.0
669            67.50       100.0
...              ...         ...
1546          115.05         0.0
1547          115.10         0.0
1548          115.15         0.0
1549          115.25         0.0
1378          105.15         0.0

[1585 rows x 2 columns]



--- TotalCharges ---
      TotalCharges  churn_rate
6529       8684.80       100.0
14           19.60       100.0
6475       8127.60       100.0
6472       8109.80       100.0
1729        563.65       100.0
...            ...         ...
6507       8375.05         0.0
6506       8349.70         0.0
6505       8349.45         0.0
6504       8337.45         0.0
6498       8312.40         0.0

[6531 rows x 2 columns]


In [17]:
df["Churn_num"] = (df["Churn"] == "Yes").astype(int)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["monthly_segment"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0, 35, 70, 90, 120],
    labels=["0-35", "35-70", "70-90", "90+"],
    include_lowest=True
)

monthly_impact = (
    df.groupby("monthly_segment")
    .agg(
        clients=("Churn", "count"),
        churners=("Churn_num", "sum"),
        churn_rate=("Churn_num", "mean"),
        revenue_at_risk=("MonthlyCharges", lambda x: x[df.loc[x.index, "Churn"] == "Yes"].sum()),
        total_charges_churners=("TotalCharges", lambda x: x[df.loc[x.index, "Churn"] == "Yes"].sum())
    )
    .reset_index()
)

monthly_impact["churn_rate"] *= 100

In [18]:
import plotly.express as px

fig = px.bar(
    monthly_impact,
    x="monthly_segment",
    y="churn_rate",
    text=monthly_impact["churn_rate"].round(1).astype(str) + "%",
    labels={
        "monthly_segment": "MonthlyCharges",
        "churn_rate": "Taux de churn (%)"
    },
    title="Taux de churn par tranche de MonthlyCharges"
)

fig.show()

In [19]:
fig = px.bar(
    monthly_impact,
    x="monthly_segment",
    y="revenue_at_risk",
    text=monthly_impact["revenue_at_risk"].round(0),
    labels={
        "monthly_segment": "MonthlyCharges",
        "revenue_at_risk": "Revenu mensuel à risque"
    },
    title="Revenu mensuel à risque par tranche de MonthlyCharges"
)

fig.show()

ANALYSE:
- Le churn est particulièrement élevé chez les clients récents. Les clients ayant entre 1 et 12 mois d’ancienneté ont un taux de churn de 47.68%, largement supérieur au taux global de 26.54%. À l’inverse, les clients les plus anciens, entre 61 et 72 mois, ont un taux de churn beaucoup plus faible, à 6.6%. Cela suggère que la première année est critiques.

- Le type de contrat est très discriminant. Les clients en Month-to-month churnent beaucoup plus que la moyenne. Les contrats longs, surtout Two year, sont très stables.

- Les clients qui paient par Electronic check ont un churn très élevé. Les paiements automatiques sont associés à un churn plus faible

- L’absence de sécurité en ligne est fortement associée au churn. Les clients avec OnlineSecurity = No sont beaucoup plus à risque.


L’analyse du taux de churn montre que certaines variables sont fortement liées au risque de résiliation. La variable tenure est particulièrement importante : les clients récents, surtout ceux ayant entre 1 et 12 mois d’ancienneté, présentent un taux de churn très élevé, autour de 47.68%, contre 26.54% en moyenne. À l’inverse, les clients les plus anciens, entre 61 et 72 mois, churnent beaucoup moins, avec un taux d’environ 6.6%.

Le type de contrat renforce fortement cette tendance : les clients en contrat mensuel ont un taux de churn de 42.71%, alors que les contrats d’un an et deux ans sont beaucoup plus stables, avec respectivement 11.27% et 2.83% de churn. Le moyen de paiement est également discriminant, notamment pour les clients utilisant l’Electronic check, qui présentent un taux de churn de 45.29%. Enfin, l’absence de service OnlineSecurity est associée à un risque élevé, avec 41.77% de churn.

En revanche, PhoneService et MultipleLines semblent moins explicatives prises isolément, car leurs taux de churn sont proches du taux global.

Globalement, les clients les plus à risque sont donc plutôt des clients récents, avec un contrat mensuel, payant par Electronic check, et ne disposant pas d’OnlineSecurity. Ces profils constituent une cible prioritaire pour les actions marketing et de rétention, surtout dans un contexte de budget limité.


L’analyse par tranche de `MonthlyCharges` permet de mieux comprendre l’impact financier du churn. Les clients avec des charges mensuelles faibles, entre 0 et 35, ont un taux de churn limité à 10.89%. En revanche, les tranches 70-90 et 90+ présentent des taux de churn beaucoup plus élevés, respectivement 37.80% et 32.78%.

Ces deux segments sont particulièrement importants car ils représentent aussi le revenu mensuel à risque le plus élevé. Les clients churners de la tranche 70-90 représentent environ 55 707 de revenu mensuel perdu potentiel, tandis que ceux de la tranche 90+ représentent environ 56 784.

Le graphique basé sur `TotalCharges` doit être interprété avec prudence. `TotalCharges` correspond au montant déjà payé par le client dans le passé : ce n’est donc pas une perte future directe. En revanche, il permet d’identifier les clients à forte valeur historique. Les clients churners de la tranche 90+ cumulent environ 1 773 730 de `TotalCharges`, ce qui montre qu’une partie importante du churn concerne aussi des clients qui ont déjà généré beaucoup de revenu.

En conclusion, les actions de rétention ne doivent pas seulement cibler les clients avec la plus forte probabilité de churn, mais aussi ceux qui ont une forte valeur mensuelle. Les segments `MonthlyCharges` 70-90 et 90+ sont donc prioritaires, surtout lorsqu’ils se combinent avec d’autres facteurs de risque : faible `tenure`, contrat mensuel, paiement par `Electronic check` ou absence de `OnlineSecurity`.


Definition du protocole d'evalutaion
   
    Split : train / validation / test (stratifié  Churn) => 10% pui 80/20 sur le reste
    Métriques :  ROC-AUC, matrice de confusion, precision/rappel/F1 et precision . Pourquoi precision: il y a un budget. Il faut cibler la population qui a le plus de chance de partir
    Déséquilibre : 25 % de churn =1 contre 75% churn =0